<a href="https://colab.research.google.com/github/Shineii86/GitHost/blob/main/notebooks/GitHost.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div align="center">
  <img src="https://raw.githubusercontent.com/Shineii86/GitHost/main/images/GitHost.png" width="300px" alt="GitHost Pro">
  <h1>📸 GitHost Pro v2.0</h1>
  <p><b>The ultimate self-hosted media sharing platform — now modular &amp; improved.</b></p>
</div>

---

> 🆕 **v2.0 Changes**
> - Refactored into modular Python package
> - Beautiful dark-themed UI (gallery, admin, error pages)
> - Admin session authentication (no password in every request)
> - Path traversal protection on static files
> - JSON API endpoints (`/api/stats`, `/api/links`)
> - Storage stats dashboard on home page
> - Better error handling & logging
> - Configurable admin password via env var
> - Streaming URL downloads (large file support)
> - RGBA→RGB thumbnail conversion
> - Download all exports original filenames
> - Manual expired cleanup button in admin

---

In [ ]:
#@title 📦 1. Install Dependencies & Clone Repo
!pip install -q GitPython PyGithub Flask flask-cors pyngrok Pillow qrcode[pil] requests python-telegram-bot

import os
import sys
import shutil

# Clone the repo to get the githost package
REPO_URL = "https://github.com/Shineii86/GitHost.git"
CLONE_DIR = "/content/GitHost"

if os.path.exists(CLONE_DIR):
    shutil.rmtree(CLONE_DIR)

!git clone $REPO_URL $CLONE_DIR
sys.path.insert(0, CLONE_DIR)

from githost.config import AppConfig
from githost.storage import GitHubStorage
from githost.notifications import EmailNotifier, TelegramNotifier
from githost.shortener import BitlyShortener
from githost.cleanup import cleanup_expired
from githost.app import create_app

print("✅ All dependencies installed and modules loaded.")

In [ ]:
#@title ⚙️ 2. Configuration & Run
import threading
import logging
from pyngrok import ngrok, conf
from IPython.display import display, Markdown, Image as IPImage
from google.colab import files
import qrcode

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(name)s: %(message)s")
logger = logging.getLogger("githost")

# =============================================================================
#  🔧 Core Configuration
# =============================================================================

GITHUB_USERNAME = "Shineii86"  #@param {type:"string"}
GITHUB_TOKEN = "ghp_xxxxxxxxxxxxxxxxxxxx"  #@param {type:"string"}
REPO_NAME = "my-image-hosting"  #@param {type:"string"}
NGROK_AUTH_TOKEN = ""  #@param {type:"string"}

# -----------------------------------------------------------------------------
#  🔗 Link Defaults
# -----------------------------------------------------------------------------
DEFAULT_EXPIRY_DAYS = 7  #@param {type:"integer"}
DEFAULT_MAX_VIEWS = 0  #@param {type:"integer"} (0 = unlimited)

# -----------------------------------------------------------------------------
#  📧 Email Notification (Optional)
# -----------------------------------------------------------------------------
SEND_EMAIL = False  #@param {type:"boolean"}
EMAIL_TO = "your-email@gmail.com"  #@param {type:"string"}
EMAIL_FROM = "sender@gmail.com"  #@param {type:"string"}
EMAIL_PASSWORD = "your-app-password"  #@param {type:"string"}
SMTP_SERVER = "smtp.gmail.com"  #@param {type:"string"}
SMTP_PORT = 465  #@param {type:"integer"}

# -----------------------------------------------------------------------------
#  🤖 Telegram Bot (Optional)
# -----------------------------------------------------------------------------
TELEGRAM_BOT_TOKEN = ""  #@param {type:"string"}
TELEGRAM_CHAT_ID = ""  #@param {type:"string"}

# -----------------------------------------------------------------------------
#  🔗 Bit.ly (Optional)
# -----------------------------------------------------------------------------
BITLY_ACCESS_TOKEN = ""  #@param {type:"string"}

# -----------------------------------------------------------------------------
#  🌐 ngrok Custom Subdomain (Paid Plan Only)
# -----------------------------------------------------------------------------
NGROK_SUBDOMAIN = ""  #@param {type:"string"}

# -----------------------------------------------------------------------------
#  🔐 Admin Password
# -----------------------------------------------------------------------------
ADMIN_PASSWORD = "admin123"  #@param {type:"string"}

# =============================================================================
#  📸 GitHost Pro v2.0 — Setup & Run
# =============================================================================

print(f"\n📸 GitHost Pro v2.0 — Self-Hosted Sharing")
print(f"User: {GITHUB_USERNAME} | Repo: {REPO_NAME}")
print("="*50)

# Build config from form values
import os
os.environ["GITHUB_USERNAME"] = GITHUB_USERNAME
os.environ["GITHUB_TOKEN"] = GITHUB_TOKEN
os.environ["REPO_NAME"] = REPO_NAME
os.environ["NGROK_AUTH_TOKEN"] = NGROK_AUTH_TOKEN
os.environ["NGROK_SUBDOMAIN"] = NGROK_SUBDOMAIN
os.environ["DEFAULT_EXPIRY_DAYS"] = str(DEFAULT_EXPIRY_DAYS)
os.environ["DEFAULT_MAX_VIEWS"] = str(DEFAULT_MAX_VIEWS)
os.environ["SEND_EMAIL"] = str(SEND_EMAIL)
os.environ["EMAIL_TO"] = EMAIL_TO
os.environ["EMAIL_FROM"] = EMAIL_FROM
os.environ["EMAIL_PASSWORD"] = EMAIL_PASSWORD
os.environ["SMTP_SERVER"] = SMTP_SERVER
os.environ["SMTP_PORT"] = str(SMTP_PORT)
os.environ["TELEGRAM_BOT_TOKEN"] = TELEGRAM_BOT_TOKEN
os.environ["TELEGRAM_CHAT_ID"] = TELEGRAM_CHAT_ID
os.environ["BITLY_ACCESS_TOKEN"] = BITLY_ACCESS_TOKEN
os.environ["ADMIN_PASSWORD"] = ADMIN_PASSWORD

config = AppConfig.from_env()

# Setup storage
storage = GitHubStorage(
    username=config.github.username,
    token=config.github.token,
    repo_name=config.github.repo_name,
)
storage.setup()

# Cleanup expired
removed = cleanup_expired(storage.links_db, storage.media_dir, storage.thumbs_dir)
if removed:
    storage.save_links_db()

# Setup notifiers
email_notifier = EmailNotifier(
    enabled=config.email.enabled,
    sender=config.email.sender,
    to=config.email.to,
    password=config.email.password,
    smtp_server=config.email.smtp_server,
    smtp_port=config.email.smtp_port,
)
telegram_notifier = TelegramNotifier(
    bot_token=config.telegram.bot_token,
    chat_id=config.telegram.chat_id,
)
shortener = BitlyShortener(config.bitly.access_token)

# Create app
app = create_app(
    storage=storage,
    config=config,
    notifier_email=email_notifier,
    notifier_telegram=telegram_notifier,
    shortener=shortener,
)

# Start Flask
def run_flask():
    app.run(host=config.server.host, port=config.server.port, debug=False)

thread = threading.Thread(target=run_flask)
thread.daemon = True
thread.start()

# ngrok tunnel
if NGROK_AUTH_TOKEN:
    conf.get_default().auth_token = NGROK_AUTH_TOKEN
    if NGROK_SUBDOMAIN:
        public_url = ngrok.connect(config.server.port, subdomain=NGROK_SUBDOMAIN).public_url
    else:
        public_url = ngrok.connect(config.server.port).public_url
else:
    public_url = f"http://localhost:{config.server.port}"
    print("⚠️ No ngrok token — server only accessible locally.")

print(f"\n{'='*50}")
print(f"✅ GitHost Pro v2.0 is LIVE!")
print(f"🔗 Base URL: {public_url}")
print(f"🖼️ Gallery:  {public_url}/gallery")
print(f"🔐 Admin:    {public_url}/admin (password: {ADMIN_PASSWORD})")
print(f"📦 Download: {public_url}/download_all")
print(f"📊 Stats:    {public_url}/api/stats")
print(f"\n💡 Upload files using the cell below.")
print(f"---")
print(f"📸 GitHost Pro By [Shinei Nouzen](https://github.com/Shineii86/GitHost)")

In [ ]:
#@title 📤 3. Upload Files
import uuid
from datetime import datetime, timedelta
from pathlib import Path
from githost.uploads import process_upload_from_file, process_upload_from_url, create_link_entry

print("\n📤 Upload options:")
upload_choice = input("Type 'file' to upload from storage or 'url' to upload from URL: ").strip().lower()

uploaded_results = []

if upload_choice == 'file':
    print("📁 Select file(s) from your device...")
    uploaded = files.upload()
    for filename, data in uploaded.items():
        result = process_upload_from_file(data, filename, storage.media_dir, storage.thumbs_dir)
        if result:
            uploaded_results.append(result)
            print(f"  ✅ {filename}")
        else:
            print(f"  ⚠️ Skipped {filename}")

elif upload_choice == 'url':
    url = input("Enter the media URL: ").strip()
    result = process_upload_from_url(url, storage.media_dir, storage.thumbs_dir)
    if result:
        uploaded_results.append(result)
        print(f"  ✅ Downloaded: {result.original_name}")
    else:
        print("  ❌ URL download failed")

else:
    print("❌ Invalid choice. Type 'file' or 'url'.")

if not uploaded_results:
    print("❌ No files uploaded.")
else:
    # Configure batch
    print("\n⚙️ Configure this batch:")
    expiry_input = input(f"Expiry in days (default {DEFAULT_EXPIRY_DAYS}): ").strip()
    expiry_days = int(expiry_input) if expiry_input else DEFAULT_EXPIRY_DAYS
    max_views_input = input(f"Max views (0=unlimited, default {DEFAULT_MAX_VIEWS}): ").strip()
    max_views = int(max_views_input) if max_views_input else DEFAULT_MAX_VIEWS
    password = input("Password (optional, press Enter to skip): ").strip() or None

    # Create link entries
    for result in uploaded_results:
        entry = create_link_entry(result, expiry_days, max_views, password)
        storage.links_db[result.link_id] = entry

    # Commit and push
    storage.commit_media()
    storage.save_links_db()

    # Display results
    print(f"\n{'='*50}")
    print(f"📸 Shareable Links:")
    for result in uploaded_results:
        share_url = f"{public_url}/i/{result.link_id}"
        if password:
            share_url += "?pwd=YOUR_PASSWORD"
        short_url = shortener.shorten(share_url)
        print(f"  {result.original_name}: {short_url}")
        # QR code
        qr = qrcode.make(short_url)
        qr_path = f"/content/qr_{result.link_id}.png"
        qr.save(qr_path)
        display(IPImage(qr_path, width=150))

    # Notifications
    telegram_notifier.send(f"📸 New upload on GitHost Pro!\n{len(uploaded_results)} file(s) uploaded.\nGallery: {public_url}/gallery")
    email_notifier.send("GitHost Pro Upload", f"{len(uploaded_results)} file(s) uploaded.\nGallery: {public_url}/gallery")

    print(f"\n🖼️ Gallery: {public_url}/gallery")
    print(f"🔐 Admin:   {public_url}/admin")

---

### 🚀 All 15+ Features Included

| # | Feature | Description |
|---|---------|-------------|
| 1 | One-Time View | Set `max_views=1` for self-destruct links |
| 2 | Password Protection | Optional password per batch |
| 3 | Multi-format | Images, videos, PDFs, ZIPs, audio, docs |
| 4 | Custom Expiry | Per-batch expiry days |
| 5 | Admin Dashboard | `/admin` with session auth & delete |
| 6 | QR Codes | Generated for each link |
| 7 | Thumbnails | Auto-generated for images |
| 8 | Upload from URL | Paste URL instead of file |
| 9 | Email Notifications | On upload and access |
| 10 | Bit.ly Shortening | Clean, short links |
| 11 | Public Gallery | `/gallery` with dark theme |
| 12 | Auto-Cleanup | Deletes expired files |
| 13 | Custom ngrok Subdomain | For paid ngrok users |
| 14 | Download All as ZIP | `/download_all` with original names |
| 15 | Telegram Bot | Notifications on Telegram |
| 16 | JSON API | `/api/stats` and `/api/links` |
| 17 | Path Traversal Protection | Secure static file serving |
| 18 | Storage Dashboard | Stats on home page |

---

<div align="center">

**Copyright [Shinei Nouzen](https://github.com/Shineii86) All Rights Reserved.**  
*Made with ❤️ for ultimate self-hosted sharing*

***Found this useful? Give it a Star! [Shineii86/GitHost](https://github.com/Shineii86/GitHost)***
</div>